In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import re
import seaborn as sns
import nflreadpy as data
import itertools
import polars as pl

#Intially when I loaded in nfl.load_pbp I didn't realize that it was returning stuff in Polars DataFrame so I intially 
#though to change it over to Pandas, but after learning that in this case Polars was better for large datasets I stuck to using it

pbp = data.load_pbp()
#print(pbp.columns)

#Before I started I tried to grab as many relevant variables that I thought I needed
'''pass_length,pass_location,air_yards,play_type,desc,side_of_field,yardline_100', 
posteam,interception,pass_touchdown,qtr,receiver_player_name,complete_pass'''

pbp = pbp.with_columns(pl.col('desc').str.replace(r'\xa0', ' ').alias('desc'))

Bo_Nix_Passes = pbp.filter(pl.col('desc').str.contains('10-B.Nix', literal=True) & (pl.col('play_type').str.contains('pass')))
Bo_Nix_Passes = Bo_Nix_Passes.select(['pass_length','pass_location','air_yards','play_type','desc','side_of_field','yardline_100', 
                    'posteam','interception','pass_touchdown','qtr','receiver_player_name','complete_pass', 'week'])
Bo_Nix_Passes.write_csv("Bo Nix.csv")

'''Bo_Nix_Passes = Bo_Nix_Passes.with_columns(
    pl.when(pl.col("desc").str.to_lowercase().str.contains("sack"))
    .then(pl.lit(None))
    .when(pl.col("desc").str.to_lowercase().str.contains("two-point conversion"))
    .then(pl.lit("Short"))
    .when(pl.col("pass_length").str.to_lowercase().str.contains("short"))
    .then(pl.lit("Short"))
    .when(pl.col("pass_length").str.to_lowercase().str.contains("deep"))
    .then(pl.lit("Deep"))
    .otherwise(pl.lit(None))
    .alias("throw_class")
)

Bo_Nix_Passes.select("throw_class").value_counts()'''

#After a lot of goofing around in polars I used some parts of it to intialize Bo's first 8 weeks of 2025. However afterwards
#I began to use the csv I created for data analysis

Bo_Nix_pbp = pd.read_csv("Bo Nix.csv")

def pass_length(desc):
    if pd.isna(desc) or not isinstance(desc, str):
        return "Unknown"
    if "TWO-POINT CONVERTION ATTEMPTED" in desc:
        return "Short"
    elif "short" in desc:
        return "Short"
    elif "deep" in desc:
        return "Deep"
Bo_Nix_pbp['Pass Length'] = Bo_Nix_pbp['pass_length'].apply(pass_length)

def pass_location(pass_location):
    if pd.isna(pass_location) or not isinstance(pass_location, str):
        return "Unknown"
    if "left" in pass_location:
        return "Left"
    elif "right" in pass_location:
        return "Right"
    elif "middle" in pass_location:
        return "Middle"
Bo_Nix_pbp['Pass Location'] = Bo_Nix_pbp['pass_location'].apply(pass_location)

def completion(complete_pass):
    if complete_pass == 1:
        return 1
    else:
        return 0
Bo_Nix_pbp['completion'] = Bo_Nix_pbp['complete_pass'].apply(completion)

depths = ['Short', 'Deep']
location = {'Left', 'Middle', 'Right'}


def create_heatmap_chart(Bo_Nix_pbp, title="Bo Nix Heatmaps"):
    fig = go.Figure()  #Start fresh
    
    #Normalize data
    Bo_Nix_pbp = Bo_Nix_pbp.copy()
    Bo_Nix_pbp['pass_length'] = Bo_Nix_pbp['pass_length'].str.lower().fillna('unknown')
    Bo_Nix_pbp['pass_location'] = Bo_Nix_pbp['pass_location'].str.lower().fillna('unknown')

    #Field Background
    fig.add_shape(
        type="rect",
        x0=0, y0=0, x1=53.3, y1=60,
        fillcolor="#196f0c",  #Green field
        line=dict(width=0),
        layer="below"
    )
    
    #Sidelines
    fig.add_shape(
        type="rect",
        x0=0, y0=0, x1=53.3, y1=60,
        line=dict(color="white", width=3),
        fillcolor="rgba(0,0,0,0)"
    )
    
    #Yard lines every 10 yards
    for yard in [10, 20, 30, 40, 50, 60]:
        fig.add_shape(
            type="line",
            x0=0, y0=yard, x1=53.3, y1=yard,  # Fixed: was x0=0, y0=0
            line=dict(color="white", width=2)
        )
        
        #Yard numbers on sides
        fig.add_annotation(
            x=3, y=yard,
            text=str(yard),
            showarrow=False,
            font=dict(color="white", size=10, family="Times New Roman"),
            textangle=0
        )
        fig.add_annotation(
            x=50.3, y=yard,
            text=str(yard),
            showarrow=False,
            font=dict(color="white", size=10, family="Times New Roman"),
            textangle=0
        )
    
    #Hash Marks
    for yard in range(0, 61, 5):
        # Left Hash Marks
        fig.add_shape(
            type="line",
            x0=18.5, y0=yard, x1=19.5, y1=yard,
            line=dict(color="white", width=1)
        )
        #Right Hash Marks
        fig.add_shape(
            type="line",
            x0=33.8, y0=yard, x1=34.8, y1=yard,
            line=dict(color="white", width=1)
        )
    
    #LOS
    los_position = 10
    fig.add_shape(
        type="line",
        x0=0, y0=los_position, x1=53.3, y1=los_position,
        line=dict(color="#4169E1", width=5),  # Bright blue
        layer="above"
    )
    
    #LOS text
    fig.add_annotation(
        x=26.65, y=los_position + 1.5,
        text="<b>LOS</b>",
        showarrow=False,
        font=dict(color="#4169E1", size=16, family="Times New Roman"),
        bgcolor="rgba(0,0,0,0.8)",
        borderpad=5
    )
    
    #Define zones
    short_zone_start = 10
    short_zone_end = 30
    deep_zone_start = 30
    deep_zone_end = 60

    zones = {
        ('short', 'left'): {'x': [0, 17.77], 'y': [short_zone_start, short_zone_end]},
        ('short', 'middle'): {'x': [17.77, 35.53], 'y': [short_zone_start, short_zone_end]},
        ('short', 'right'): {'x': [35.53, 53.3], 'y': [short_zone_start, short_zone_end]},
        ('deep', 'left'): {'x': [0, 17.77], 'y': [deep_zone_start, deep_zone_end]},
        ('deep', 'middle'): {'x': [17.77, 35.53], 'y': [deep_zone_start, deep_zone_end]},
        ('deep', 'right'): {'x': [35.53, 53.3], 'y': [deep_zone_start, deep_zone_end]}
    }
    
    #Calculate zone data
    zone_data = {}
    for (depth, location), bounds in zones.items():
        zone_plays = Bo_Nix_pbp[
            (Bo_Nix_pbp['pass_length'] == depth) &
            (Bo_Nix_pbp['pass_location'] == location)
        ]
        count = len(zone_plays)
        #print(zone_plays)
        
        #Receiver breakdown
        receiver_counts = zone_plays['receiver_player_name'].value_counts()
        receiver_text = "<br>".join([f"{name}: {cnt}" for name, cnt in receiver_counts.head(5).items()])
        
        zone_data[(depth, location)] = {
            'count': count,
            'receivers': receiver_text if receiver_text else "No Targets"
        }
    
    max_count = max([data['count'] for data in zone_data.values()]) if zone_data else 1
    
    #Colored zones
    for (depth, location), bounds in zones.items():
        data = zone_data.get((depth, location), {'count': 0, 'receivers': 'No targets'})
        count = data['count']

        #Color gradient
        intensity = count / max_count if max_count > 0 else 0
        if intensity < 0.5:
            red = 255
            green = int(255 * (intensity * 2))
            blue = 0
        else:
            red = int(255 * (1 - (intensity - 0.5) * 2))
            green = 255
            blue = 0
        
        color = f'rgba({red}, {green}, {blue}, 0.7)'

        center_x = (bounds['x'][0] + bounds['x'][1]) / 2
        center_y = (bounds['y'][0] + bounds['y'][1]) / 2

        def completion(complete_pass):
            if complete_pass == 1:
                return 1
            else:
                return 0
            

        #Hover point
        fig.add_trace(go.Scatter(
            x=[center_x],
            y=[center_y],
            mode = "markers",
            marker=dict(size=0.1, opacity=0),
            hovertemplate=(
                f"<b>{depth.upper()} {location.upper()}</b><br>"
                f"<b>Total Targets: {count}</b><br>"
                f"<b>Top Receivers:</b><br>"
                f"{data['receivers']}"
                "<extra></extra>"
            ),
            hoverlabel = dict(
                bgcolor = "orange",
                font_size = 14,
                font_family = "Times New Roman",
                font_color = "white"
            ),
            showlegend=False  
        ))
        
        #Zone rectangle
        fig.add_shape(
            type="rect",
            x0=bounds['x'][0], y0=bounds['y'][0],
            x1=bounds['x'][1], y1=bounds['y'][1],
            fillcolor=color,
            line=dict(color='white', width=2),
            layer="below"
        )
        
        #Annotation
        fig.add_annotation(
            x=center_x,
            y=center_y,
            text=f"<b>{count}</b><br>{depth.upper()}<br>{location.upper()}",
            showarrow=False,
            font=dict(size=16, color='white', family='Arial Black'),
            bgcolor='rgba(0,0,0,0.7)',
            borderpad=10
        )
    
    weeks = sorted(Bo_Nix_pbp['week'].unique())
    total_attempts = len(Bo_Nix_pbp)
    
    fig.update_layout(
        title={
            'text': f"{title}<br><sub>Weeks {min(weeks)}-{max(weeks)} | {total_attempts} Total Targets</sub>",
            'x': 0.5,
            'xanchor': 'center',
            'font': {'size': 24, 'color': 'white', 'family': 'Times New Roman'}
        },
        xaxis=dict(range=[-2, 55.3], showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(range=[-2, 62], showgrid=False, zeroline=False, showticklabels=False,
                   scaleanchor="x", scaleratio=1),
        plot_bgcolor='#1a1a1a',
        paper_bgcolor='#1a1a1a',
        font=dict(color='white'),
        width=600,
        height=800,
        hovermode='closest'
    )
    return fig
for week in sorted(Bo_Nix_pbp['week'].unique()):
    week_data = Bo_Nix_pbp[Bo_Nix_pbp['week'] == week]
    if len(week_data) > 0:
        fig = create_heatmap_chart(week_data, f"Bo Nix Field Zones - Week {week}")
        fig.write_html(f'bo_nix_field_zone_week_{week}_2025.html')


fig1 = create_heatmap_chart(Bo_Nix_pbp, "Bo Nix Target Distribution - 2025")
fig1.write_html('bo_nix_categorical_heatmap_2025.html')